# PC to z to CAD

In this notebook I want to develop the pipeline from point cloud to CAD-model. To do that I will use my trained PointNet++ to encode a point cloud into the latent represenation z and then the DeepCAD decoder will reconstruct the CAD model from this.

In [3]:
import sys
import importlib
import os
import torch
import open3d as o3d

sys.path.append(os.path.abspath("../code"))
sys.path.append(os.path.abspath(".."))
import dataset
from dataset import PointCloudEmbeddingDataset
importlib.reload(dataset)

<module 'dataset' from '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/code/dataset.py'>

In [4]:
print(sys.path)

['/opt/anaconda3/lib/python312.zip', '/opt/anaconda3/lib/python3.12', '/opt/anaconda3/lib/python3.12/lib-dynload', '', '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/pointnet.pytorch/pointnet_venv/lib/python3.12/site-packages', '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/code', '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction/code', '/Users/saidharb/Documents/LocalDocuments/Master-Thesis/Point-Cloud-Reconstruction']


In [5]:
def inplace_relu(m):
    classname = m.__class__.__name__
    if classname.find('ReLU') != -1:
        m.inplace=True

In [6]:
train_dataset = PointCloudEmbeddingDataset("../data", 'train')

Loading train dataset 

Number of samples that should be in the train set: 161240
Files on disk: 160982 --> There are 258 missing point cloud files in the train set.

Checking latent representation:
All latent represenations are valid.

--- DONE ---



In [21]:
import random
random.randint(0, 2)

2

In [4]:
index = 0

point_cloud = train_dataset[index][0]
latent_rep = train_dataset[index][1]
train_dataset.get_path(index)

'../data/pc_cad/0067/00675619.ply'

In [5]:
def visualize_pc(pc_path):
    point_cloud = o3d.io.read_point_cloud(pc_path)
    print(point_cloud)
    o3d.visualization.draw_geometries([point_cloud])
#visualize_pc(train_dataset.get_path(index))

## Plan
- import trained PN++ DONE
- infer PC with PN++ DONE
- compare predicted z with target z DONE (using MSE)
- import pretrained decoder
- infer z with decoder
- compare predicted CAD-sequence with target CAD-sequence
- visualize CAD model

In [6]:
sys.path.append(os.path.join("..", 'models','Pointnet_Pointnet2_pytorch', 'models'))
model = importlib.import_module('pointnet2_cls_ssg')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

classifier = model.get_model(256, normal_channel=False)
criterion = model.get_loss_mse()
classifier.apply(inplace_relu)

model_path = 'best.pth'
model_dict = torch.load(model_path, map_location=torch.device(device), weights_only=True)
state_dict = model_dict['model_state_dict']
classifier.load_state_dict(state_dict)
classifier = classifier.to(device)
classifier.eval()

print("Loaded model")

Loaded model


In [7]:
def pc_to_z(model, pc, target):
    with torch.no_grad():
        pc = pc.unsqueeze(1) # CRITICAL: shape = (B,1,256), NOT (1,B,256) -> unsqueeze(1 not 0)
        target = target.unsqueeze(0)
        pc = pc.transpose(2,1)
        pred, _ = classifier(pc)
        mse = criterion(pred,target)
        print(f"MSE: {mse:.4f}")
        

In [8]:
pc_to_z(classifier, point_cloud, latent_rep)

MSE: 0.1229


In [16]:
sys.path.append(os.path.abspath(".."))
from models.DeepCAD.trainer.trainerAE import TrainerAE

In [ ]:
print(sys.path)

In [3]:
import h5py
hehe = '../models/trained_models/test_run/results/testing/z.h5'

In [48]:
with h5py.File(hehe, 'r') as hf:
   # data = hf['cad_sequence'][:20].tolist()
    print(hf['cad_sequence'][1])

[[  4  -1  -1 ...  -1  -1  -1]
 [  0 223 128 ...  -1  -1  -1]
 [  0 223 223 ...  -1  -1  -1]
 ...
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]
 [  3  -1  -1 ...  -1  -1  -1]]


In [32]:
print(type(data))

<class 'list'>


In [55]:
data[4]

[[4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 223, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 128, 223, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [5, -1, -1, -1, -1, -1, 128, 128, 128, 32, 130, 128, 96, 130, 128, 0, 0],
 [4, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 176, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 223, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 128, 207, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [0, 128, 128, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [5, -1, -1, -1, -1, -1, 128, 128, 128, 128, 152, 128, 144, 32, 128, 0, 0],
 [3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1],
 [3, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1, -1

In [59]:
hoho = '../data/cad_vec/0000/00000070.h5'
with h5py.File(hoho, 'r') as lol:
    print(lol['vec'])

<HDF5 dataset "vec": shape (23, 17), type "<i8">


In [61]:
import os
os.path.splitext(hoho)

('../data/cad_vec/0000/00000070', '.h5')

## Understanding the CADLoss

In [21]:
import torch.nn.functional as F
import torch

In [68]:
tgt_commands = torch.tensor(
       [[4, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 5, 4, 2, 5, 4, 2, 5, 4, 2, 5,
         4, 2, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
        [4, 0, 1, 0, 1, 5, 4, 0, 1, 5, 4, 1, 0, 5, 4, 2, 4, 2, 5, 4, 0, 0, 0, 0,
         5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]])

In [69]:
tgt_args = torch.load("example.pth", weights_only = True)

In [24]:
print(tgt_args.shape)

torch.Size([2, 60, 16])


In [25]:
#visibility_mask = torch.tensor([True, True])

In [27]:
#padding_mask = torch.tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.]])

IndentationError: unindent does not match any outer indentation level (<string>, line 5)

In [28]:
padding_mask.shape

NameError: name 'padding_mask' is not defined

In [71]:
lol = torch.load("example2.pth", weights_only = True)
command_logits = lol['command_logits']
args_logits = lol['args_logits']
mask = lol['mask']
padding_mask = lol['padding_mask']

In [31]:
command_logits[padding_mask.bool()].reshape(-1, self.n_commands)

NameError: name 'padding_mask' is not defined

In [32]:

def _get_visibility_mask(commands, seq_dim=0):
    """
    Args:
        commands: Shape [S, ...]
    """
    print("commands targets shape: ", commands.shape)
    S = commands.size(seq_dim)
    print(f"S is the last dimension: {S}")
    with torch.no_grad():
        mask1 = commands == 3 # EOS_IDX = 3
        print(f"\nFirst every command which is EOS is marked with True:")
        print(mask1)
        mask2 = mask1.sum(dim=seq_dim)
        print(f"\nThen, the number of non used commands is added up per sample: ")
        print(mask2)
        print(f"\nThen they check if there are at least 2 non EOS commands (by checking if the number of EOS_IDX is <(60-1): ")
        visibility_mask = mask2 < S - 1 
        print(visibility_mask)

        if seq_dim == 0:
            return visibility_mask.unsqueeze(-1)
        return visibility_mask

In [72]:
visibility_mask = _get_visibility_mask(tgt_commands, seq_dim = -1)

commands targets shape:  torch.Size([2, 60])
S is the last dimension: 60

First every command which is EOS is marked with True:
tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
  

In [55]:
def _get_padding_mask(commands, seq_dim=0, extended=False):
    print("Input: target commands")
    with torch.no_grad():
        print(f"\nFirst every command which is EOS is marked with True:")
        mask1 = commands == 3
        print(mask1)
        print(f"\nThen the cummulative sum of True entries across the rows is calculated:")
        mask2 = mask1.cumsum(dim = seq_dim)
        print(mask2)
        padding_mask = mask2 == 0
        print(f"\nThereby it is easy to mark every EOS command with False:")
        print(padding_mask)
        print(f"\nFinally converted to float:")
        padding_mask = padding_mask.float()
        print(padding_mask)

        if not extended:
            # padding_mask doesn't include the final EOS, extend by 1 position to include it in the loss
            S = commands.size(seq_dim)

            
            torch.narrow(padding_mask, seq_dim, 3, S-3).add_(torch.narrow(padding_mask, seq_dim, 0, S-3)).clamp_(max=1)
            #print(aha)
        if seq_dim == 0:
            return padding_mask.unsqueeze(-1)
        return padding_mask

In [73]:
print(tgt_commands)
padding_mask = _get_padding_mask(tgt_commands, seq_dim = -1, extended = True)
print("\nPadding Mask: \n", padding_mask)

tensor([[4, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 5, 4, 2, 5, 4, 2, 5, 4, 2, 5,
         4, 2, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
        [4, 0, 1, 0, 1, 5, 4, 0, 1, 5, 4, 1, 0, 5, 4, 2, 4, 2, 5, 4, 0, 0, 0, 0,
         5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]])
Input: target commands

First every command which is EOS is marked with True:
tensor([[False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False, False, False, False,
         False, False, False, False, False, False, False,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True,
          True,  True,  True,  True,  True,  True,  True,  True,  True,  True],
        [False, False, False, 

In [74]:
print("The visibility mask is used to filter non valid command sequences.")
padding_mask = padding_mask * visibility_mask.unsqueeze(-1)
padding_mask

The visibility mask is used to filter non valid command sequences.


tensor([[1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.],
        [1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 1., 1., 1., 1., 1., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0., 0.,
         0., 0., 0., 0., 0., 0.]])

In [75]:
# from macro.py
import numpy as np
ALL_COMMANDS = ['Line', 'Arc', 'Circle', 'EOS', 'SOL', 'Ext']
LINE_IDX = ALL_COMMANDS.index('Line')
ARC_IDX = ALL_COMMANDS.index('Arc')
CIRCLE_IDX = ALL_COMMANDS.index('Circle')
EOS_IDX = ALL_COMMANDS.index('EOS')
SOL_IDX = ALL_COMMANDS.index('SOL')
EXT_IDX = ALL_COMMANDS.index('Ext')
PAD_VAL = -1
N_ARGS_SKETCH = 5 # sketch parameters: x, y, alpha, f, r
N_ARGS_PLANE = 3 # sketch plane orientation: theta, phi, gamma
N_ARGS_TRANS = 4 # sketch plane origin + sketch bbox size: p_x, p_y, p_z, s
N_ARGS_EXT_PARAM = 4 # extrusion parameters: e1, e2, b, u
N_ARGS_EXT = N_ARGS_PLANE + N_ARGS_TRANS + N_ARGS_EXT_PARAM
N_ARGS = N_ARGS_SKETCH + N_ARGS_EXT

SOL_VEC = np.array([SOL_IDX, *([PAD_VAL] * N_ARGS)])
EOS_VEC = np.array([EOS_IDX, *([PAD_VAL] * N_ARGS)])

CMD_ARGS_MASK = np.array([[1, 1, 0, 0, 0, *[0]*N_ARGS_EXT],  # line
                          [1, 1, 1, 1, 0, *[0]*N_ARGS_EXT],  # arc
                          [1, 1, 0, 0, 1, *[0]*N_ARGS_EXT],  # circle
                          [0, 0, 0, 0, 0, *[0]*N_ARGS_EXT],  # EOS
                          [0, 0, 0, 0, 0, *[0]*N_ARGS_EXT],  # SOL
                          [*[0]*N_ARGS_SKETCH, *[1]*N_ARGS_EXT]]) # Extrude
cmd_args_mask = CMD_ARGS_MASK
n_commands = len(ALL_COMMANDS)

In [38]:
print(f"The authors then use a command args mask to mask invalid args per command \n{CMD_ARGS_MASK}\n{CMD_ARGS_MASK.shape}")


The authors then use a command args mask to mask invalid args per command 
[[1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0]
 [1 1 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
 [0 0 0 0 0 1 1 1 1 1 1 1 1 1 1 1]]
(6, 16)


In [76]:
mask = cmd_args_mask[tgt_commands.long()]
print("Then they access that mask using the target commands, thereby obtaining a custom mask for the commands")

Then they access that mask using the target commands, thereby obtaining a custom mask for the commands


In [62]:
tgt_commands

tensor([[4, 0, 0, 0, 4, 2, 4, 2, 4, 2, 4, 2, 4, 2, 5, 4, 2, 5, 4, 2, 5, 4, 2, 5,
         4, 2, 5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3],
        [4, 0, 1, 0, 1, 5, 4, 0, 1, 5, 4, 1, 0, 5, 4, 2, 4, 2, 5, 4, 0, 0, 0, 0,
         5, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3,
         3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3, 3]])

In [63]:
mask[0][:6]

array([[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0],
       [1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]])

### Calculation of Loss

- When you index command_logits using padding_mask.bool(), PyTorch flattens the first two dimensions (batch and sequence length) and selects only the elements where the mask is True.

In [42]:
command_logits.shape, padding_mask.bool().shape

(torch.Size([2, 60, 6]), torch.Size([2, 60]))

In [46]:
command_logits[padding_mask.bool()].reshape(-1, n_commands).shape

torch.Size([52, 6])

In [48]:
tgt_commands[padding_mask.bool()].reshape(-1).shape

torch.Size([52])

## Summary CADLoss

#### Commands
- using the target commands of shape (B, 60) a visibility (B) and padding mask (B, 60) are created
    - visibility mask: Basically checks if the samples are valid or not (are there 2 or more non EOS commands)
    - padding mask: Marks every non-EOS command
- Now when passsing the logits and targets to cross entropy, the padding mask makes sure only the commands actually belonging to the sequence are processed, and not the unused (EOS) commands

#### Args
- using the target commands of shape (B, 60) a command_args mask (B, 60, 16) is created
    - Command_args mask: Marks all valid arguments per command
- when passing the args_logits and targets to cross entropy, the command_args_mask makes sure only valid args take part in the calculation of loss
- The arg loss is multiplied by 2 as stated in macro.py



In [67]:
mask.shape, mask.sum()

((2, 60, 16), np.int64(181))

In [59]:
args_logits.shape

torch.Size([2, 60, 16, 257])

## TEST

In [101]:
lol = torch.load("example2.pth", weights_only = True)
command_logits = lol['command_logits']
args_logits = lol['args_logits']
mask = lol['mask']
padding_mask = lol['padding_mask']
tgt_commands = lol['tgt_commands']
tgt_args = lol['tgt_args']

In [110]:
command_logits.shape, tgt_commands.shape

(torch.Size([2, 60, 6]), torch.Size([2, 60]))

In [113]:
command_logits_masked = command_logits[padding_mask.bool()]
command_logits_masked.shape

torch.Size([58, 6])

In [120]:
softmax_probs = torch.softmax(command_logits_masked, dim = -1)
correct_class_probs = softmax_probs[torch.arange(command_logits_masked.shape[0]), tgt_commands[padding_mask.bool()]]

In [121]:
correct_class_probs.shape

torch.Size([58])

In [122]:
loss_manual = -torch.log(correct_class_probs).mean()

In [123]:
loss_manual #CORRECT!

tensor(21.9111)

In [143]:
tgt_args[mask.bool()].shape
tgt_args[mask.bool()]+ 1

tensor([217, 129, 175, 224, 129, 129, 142, 136,   7, 158, 135,   6, 175, 135,
          6, 190, 135,   6, 206, 136,   6, 193,  65, 193,  48, 129,  46, 168,
        149, 129,   1,   1, 177, 129,  49, 193,  65, 193,  76, 109,  78,  20,
        205, 129,   2,   1, 177, 129,  49, 193,  65, 193, 106, 109,  77,  19,
        205, 129,   2,   1, 177, 129,  49, 193,  65, 193, 132, 109,  77,  19,
        205, 129,   2,   1, 177, 129,  49, 193,  65, 193, 159, 109,  77,  19,
        205, 129,   2,   1, 219, 129, 219, 174,  39,   2, 129, 174, 129, 129,
         39,   2, 129, 129, 129, 114, 122, 129,  33,  33, 129,   1,   1, 224,
        129, 129, 129,  91,   2, 129, 129, 129, 114, 136, 129,  31,  33, 129,
          2,   1, 224, 129,  91,   2, 129, 129, 129, 129, 129, 114, 122, 129,
         31,  33, 129,   2,   1, 177, 129,  49, 177, 129,  41, 129, 129, 129,
        109, 129, 129,  41, 123, 129,   2,   1, 182, 129, 182, 224, 129, 224,
        129, 129, 193,  65, 193, 119, 129,  86,  38,  93, 129,  

In [119]:
args_logits_masked = args_logits[mask.bool()]
args_logits_masked.shape

torch.Size([181, 257])

In [130]:
softmax_probs = torch.softmax(args_logits_masked, dim = -1)
correct_class_probs = softmax_probs[torch.arange(args_logits_masked.shape[0]), tgt_args[mask.bool()] + 1]
# +1 shift because -1 was used as padding value
# the model will learn the correct label classes, because loss is calcualted after the shift
# However, why is +1 added to the targets that are valid? There are no -1 in the targets right? 
# wie dem auch sei, auch wenn es sinnlos sein sollte, lernen tut das model trotzdem richtig

In [132]:
loss_manual = -torch.log(correct_class_probs).mean()
loss_manual * 2 ## CORRECT!

tensor(24.3764)

In [2]:
a = "hefjn/cjds/ak.h5"
a.split("/")[-1]

'ak.h5'